# CLIP-Based Multimodal Retrieval

Text-to-image retrieval and multimodal score fusion over dermatology images.

## Experiment scope

- Encode images with CLIP
- Build a semantic text index over image metadata
- Build an image-vector index in ChromaDB
- Query images from natural-language text
- Combine text and image rankings with weighted fusion
- Use retrieved metadata as bounded context for an LLM response

> The dataset is used only for retrieval engineering evaluation. This is not a diagnostic system.


In [ ]:
import sys
from pathlib import Path
# Works from the repository root, experiment directory, or notebooks directory.
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((candidate for ancestor in _candidates
                     for candidate in (ancestor, ancestor / "experiments/multimodal_retrieval")
                     if (candidate / "config.yaml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from this repository or its multimodal_retrieval directory")
sys.path.insert(0, str(PROJECT_ROOT.parents[1]))
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")
load_dotenv(PROJECT_ROOT.parents[1] / ".env")

from experiments.multimodal_retrieval.src.services import (
    load_config, get_llm, get_clip_model, get_text_embeddings, validate_api_keys, print_config_summary,
)
from experiments.multimodal_retrieval.src.retrieval import (
    load_dermamnist, encode_images, collection_names, sync_text_index, sync_image_index,
    validate_search, image_search as _image_search, fused_search as _fused_search,
)
from datetime import datetime, timezone
import json
config = load_config(str(PROJECT_ROOT / "config.yaml"))
api_key_status = validate_api_keys(config)
print_config_summary(config)

In [ ]:
# Initialize retrieval models first; LLM generation is optional.
clip_model = get_clip_model(config)
text_embeddings = get_text_embeddings(config)

required_key = "OPENROUTER_API_KEY" if config["llm_provider"] == "openrouter" else "OPENAI_API_KEY"
llm = get_llm(config) if api_key_status.get(required_key, False) else None

print(f"✅ CLIP: {config['clip_model']}")
print(f"✅ Text embeddings: {config['text_emb_model']}")

if llm is not None:
    print(f"✅ LLM: {config['llm_provider']} / {config.get('openrouter_model', config.get('llm_model'))}")
    try:
        test_response = llm.invoke("Reply with exactly: API working")
        test_msg = test_response.content if hasattr(test_response, "content") else str(test_response)
        print(f"✅ LLM API verified: {test_msg[:80]}")
    except Exception as e:
        llm = None
        print(f"⚠️ LLM API test failed, generation disabled: {e}")
else:
    print(f"ℹ️ {required_key} is not configured. Retrieval cells still run; generation will be skipped.")

---

## Dataset: DermaMNIST

This experiment uses **DermaMNIST**, a seven-class dermatoscopic image dataset derived from HAM10000 and distributed through MedMNIST. The full dataset contains 10,015 images, with 7,007 in the official training split. This notebook uses the 128×128 version and a configurable subset for fast retrieval experiments.

**Official distribution:** MedMNIST / Zenodo


In [ ]:
import pandas as pd
import numpy as np
from PIL import Image

images_dir = Path(config["data_root"]) / "images"
saved_dir = images_dir / "dermamnist"
saved_dir.mkdir(parents=True, exist_ok=True)
dermamnist_path = Path(config["data_root"]) / "dermamnist_128.npz"

# Disease category mappings (7 categories in DermaMNIST from HAM10000)
label_names = {
    0: "Actinic Keratoses and Intraepithelial Carcinoma",
    1: "Basal Cell Carcinoma",
    2: "Benign Keratosis-like Lesions",
    3: "Dermatofibroma",
    4: "Melanoma",
    5: "Melanocytic Nevi",
    6: "Vascular Lesions",
}

# Short descriptive metadata is used only to support text retrieval evaluation.
label_descriptions = {
    0: "Sun-damage-associated keratinocytic lesion class that can appear rough, scaly, or erythematous in dermatoscopic imagery.",
    1: "Basal cell carcinoma lesion class, often associated with pearly, shiny, ulcerated, or vascular visual patterns.",
    2: "Benign keratosis-like lesion class, including pigmented or warty-appearing benign keratinocytic lesions.",
    3: "Dermatofibroma lesion class, typically represented by firm benign fibrous skin lesions with characteristic central and peripheral patterns.",
    4: "Melanoma lesion class, a malignant melanocytic category that may show asymmetry, irregular borders, and heterogeneous pigmentation.",
    5: "Melanocytic nevi lesion class, representing common benign melanocytic moles with a range of pigmentation and structural patterns.",
    6: "Vascular lesion class, representing benign blood-vessel-related lesions that can appear red, purple, or blue in skin imagery.",
}



images_array, labels_array, selected_indices = load_dermamnist(dermamnist_path, config["subset_n"])
items = []
for image_array, label, train_index in zip(images_array, labels_array, selected_indices):
    # Use the original training index, not the item's position in a sampled subset.
    item_id = f"derma_train_{train_index:06d}"
    path = saved_dir / f"{item_id}.png"
    Image.fromarray(image_array).save(path)
    items.append({
        "item_id": item_id, "title": label_names[int(label)],
        "description": label_descriptions[int(label)],
        "category": label_names[int(label)].lower().replace(" ", "_"),
        "img_path": str(path.relative_to(config["data_root"])), "label": int(label),
        "train_index": int(train_index),
    })
df = pd.DataFrame(items)
print(f"Loaded {len(df)} images, shape: {images_array.shape}")
print(df.head())
print(df['category'].value_counts())

---

## Image embeddings

Encode all images with CLIP's vision tower.


In [ ]:
image_embeddings, valid_indices = encode_images(df, clip_model, config["data_root"])
# valid_indices contains DataFrame labels, not positional offsets.
df = df.loc[valid_indices].copy()
print(f"Encoded {len(df)} images: {image_embeddings.shape}")

In [ ]:
image_embeddings[0]

---

## Vector indexes

Create two collections:
1. **health_text**: Text descriptions (title + description)
2. **health_image**: Image embeddings


In [ ]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
import chromadb

chroma_root = Path(config["artifacts_root"]) / "chroma"
text_collection_name, image_collection_name = collection_names(config)
text_docs, metadatas = [], []
for _, row in df.iterrows():
    metadata = {key: row[key] for key in ("item_id", "title", "category", "img_path")}
    metadatas.append(metadata)
    text_docs.append(Document(page_content=f"{row['title']}. {row['description']}", metadata=metadata))
ids = df["item_id"].tolist()

text_vectorstore = Chroma(
    collection_name=text_collection_name, embedding_function=text_embeddings,
    persist_directory=str(chroma_root / "health_text"),
    collection_metadata={"hnsw:space": "cosine"},
)
sync_text_index(text_vectorstore, text_docs, ids)
chroma_client = chromadb.PersistentClient(path=str(chroma_root / "health_image"))
image_collection = chroma_client.get_or_create_collection(
    image_collection_name, metadata={"hnsw:space": "cosine"}, embedding_function=None,
)
sync_image_index(image_collection, ids, image_embeddings, metadatas)
print(f"Indexed {len(ids)} current items in {text_collection_name} and {image_collection_name}")

---

## Retrieval modes

### Mode 1: Text-only retrieval


In [ ]:
def text_only_search(query: str, top_k: int = 3, category_filter: str = None):
    """Search using text embeddings only."""
    validate_search(query, top_k)
    retriever = text_vectorstore.as_retriever(search_kwargs={"k": top_k})
    
    # Apply filter if specified
    if category_filter:
        retriever.search_kwargs["filter"] = {"category": category_filter}
    
    results = retriever.invoke(query)
    return results

# Test
query = "pearly or waxy lesion consistent with basal cell carcinoma"
results = text_only_search(query, top_k=3)

print(f"📝 Text-only search: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.metadata['title']} ({doc.metadata['category']})")
    print(f"    {doc.page_content[:80]}...\n")

### Mode 2: Image-only retrieval


In [ ]:
def image_search(query_text: str, top_k: int = 3, category_filter=None):
    return _image_search(image_collection, clip_model, query_text, top_k, category_filter)

# Test
query = "dark irregular lesion with changing borders, possible melanoma"
results = image_search(query, top_k=3)

print(f"🖼️  Image search: '{query}'\n")
for i, (item_id, metadata) in enumerate(zip(results['ids'][0], results['metadatas'][0]), 1):
    print(f"[{i}] {metadata['title']} ({metadata['category']})")
    print(f"    {item_id}\n")

### Mode 3: Fusion (Text + Image)

Combine text and image retrieval using score fusion.


In [ ]:
def fused_search(query: str, top_k: int = 3, text_weight: float = 0.5):
    return _fused_search(text_vectorstore, image_collection, clip_model, query, top_k, text_weight)

# Test
query = "common benign mole or melanocytic nevus"
fused, t_scores, i_scores = fused_search(query, top_k=3, text_weight=0.6)

print(f"🔀 Fused search (60% text, 40% image): '{query}'\n")
for i, (item_id, score) in enumerate(fused, 1):
    row = df[df['item_id'] == item_id].iloc[0]
    t = t_scores.get(item_id, 0.0)
    im = i_scores.get(item_id, 0.0)
    print(f"[{i}] {row['title']} ({row['category']})")
    print(f"    Fused: {score:.3f} (text: {t:.3f}, image: {im:.3f})\n")

---

## Result visualization

Display top-k images in a grid.


In [ ]:
import matplotlib.pyplot as plt

def visualize_results(item_ids, scores=None, title="Search Results", ncols=3):
    """Display images in a grid."""
    if isinstance(ncols, bool) or not isinstance(ncols, int) or ncols < 1:
        raise ValueError("ncols must be a positive integer")
    if not item_ids:
        print("No results to display")
        return None
    nrows = (len(item_ids) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3, nrows*3))
    
    # Handle axes array properly for different grid sizes
    if nrows == 1 and ncols == 1:
        axes = [axes]  # Single subplot
    elif nrows == 1 or ncols == 1:
        axes = axes if isinstance(axes, np.ndarray) else [axes]
    else:
        axes = axes.flatten()  # 2D grid
    
    for i, item_id in enumerate(item_ids):
        row = df[df['item_id'] == item_id].iloc[0]
        img_path = Path(config["data_root"]) / row["img_path"]
        with Image.open(img_path) as opened:
            img = opened.convert("RGB")
        
        axes[i].imshow(img)
        axes[i].axis('off')
        
        score_str = f" ({scores[item_id]:.3f})" if scores else ""
        axes[i].set_title(f"[{i+1}] {row['title']}{score_str}", fontsize=9)
    
    # Hide unused subplots
    for i in range(len(item_ids), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize fused results
visualize_results(
    [item_id for item_id, _ in fused],
    scores={item_id: score for item_id, score in fused},
    title=f"Fused Search: '{query}'"
)

---

## Retrieval-grounded generation

Use retrieved items as context for LLM generation.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def multimodal_rag(query: str, top_k: int = 3):
    """RAG using multimodal retrieval."""
    # Retrieve
    fused_results, _, _ = fused_search(query, top_k=top_k)
    
    if not fused_results:
        return "No relevant items were retrieved.", []

    # Build context
    context_parts = []
    for i, (item_id, score) in enumerate(fused_results, 1):
        row = df[df['item_id'] == item_id].iloc[0]
        context_parts.append(f"[{i}] {row['title']} ({row['category']}): {row['description']}")
    
    context = "\n".join(context_parts)
    
    # RAG prompt
    prompt = ChatPromptTemplate.from_template(
        """You are a retrieval-grounded health information assistant. Use ONLY the provided context to answer.
This is for research and engineering purposes only, NOT medical diagnosis or advice.

Context:
{context}

Question: {question}

Answer (keep under 6 sentences):"""
    )
    
    if llm is None:
        raise RuntimeError("Configure the LLM API key before running generation.")

    chain = prompt | llm
    response = chain.invoke({"context": context, "question": query})
    answer = response.content if hasattr(response, "content") else str(response)
    
    return answer, fused_results

# Optional retrieval-grounded generation
query = "What does the retrieved context say about basal cell carcinoma?"

if llm is not None:
    answer, results = multimodal_rag(query, top_k=3)

    print(f"❓ Query: {query}\n")
    print("📚 Retrieved items:")
    for i, (item_id, score) in enumerate(results, 1):
        row = df[df["item_id"] == item_id].iloc[0]
        print(f"  [{i}] {row['title']} (score: {score:.3f})")

    print(f"\n💡 Answer:\n{answer}")
else:
    print("Skipping optional generation because no LLM API key is configured.")

---

## Save Manifest


In [ ]:
manifests_dir = Path(config["artifacts_root"]) / "manifests"
manifests_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "text_collection_name": text_collection_name,
    "image_collection_name": image_collection_name,
    "clip_model": config["clip_model"],
    "text_emb_model": config["text_emb_model"],
    "num_items": len(df),
    "categories": df['category'].unique().tolist(),
    "created_at": datetime.now(timezone.utc).isoformat(),
}

manifest_path = manifests_dir / "health.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"✅ Manifest saved: {manifest_path}")

---

## Summary

**Implemented:**
- CLIP image and text embeddings in a shared representation space
- Separate text and image collections
- Text-only, image-only, and fused retrieval
- Weighted score fusion across modalities
- Retrieval-grounded generation from returned metadata

**Implemented fusion:**
- **Weighted fusion**: `score = w₁·text_score + w₂·image_score`

**Alternative fusion:**
- Reciprocal Rank Fusion (RRF) can provide rank-based aggregation across retrieval channels.

**Fusion considerations:**
- Queries that combine textual and visual cues
- Datasets with complementary text and image signals
- Retrieval workloads where a single modality is insufficient

**Limitations:**
- Zero-shot CLIP embeddings may miss domain-specific distinctions
- Fusion weights require tuning for the target retrieval distribution
- Global image embeddings can miss fine-grained visual details

**Artifacts:**
- `./artifacts/chroma/health_text/`
- `./artifacts/chroma/health_image/`
- `./artifacts/manifests/health.json`
